In [1]:
from zipfile import ZipFile
import pandas as pd

GTFS_ZIP = "sweden_api.zip"

with ZipFile(GTFS_ZIP) as z:
    agencies = pd.read_csv(z.open("agency.txt"), dtype=str)
    routes   = pd.read_csv(z.open("routes.txt"), dtype=str)
    trips    = pd.read_csv(z.open("trips.txt"), dtype=str)

print("Agencies:", len(agencies))
print("Routes:", len(routes))
print("Trips:", len(trips))

Agencies: 249
Routes: 7575
Trips: 429244


In [5]:
import requests
import os

from google.transit import gtfs_realtime_pb2

API_KEY = os.environ["GTFSSweden3"]

url = "https://opendata.samtrafiken.se/gtfs-rt-sweden/sl/VehiclePositions.pb"

r = requests.get(url, params={"key": API_KEY})
r.raise_for_status()

feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(r.content)

print("Entities:", len(feed.entity))

Entities: 891


In [6]:
import pandas as pd

rows = []

for entity in feed.entity:

    if not entity.HasField("vehicle"):
        continue

    v = entity.vehicle

    rows.append({
        "vehicle_id": v.vehicle.id,
        "trip_id": v.trip.trip_id,
        "route_id": v.trip.route_id,
        "lat": v.position.latitude,
        "lon": v.position.longitude,
        "timestamp": v.timestamp
    })

realtime = pd.DataFrame(rows)

print(realtime.head())
print(len(realtime))

         vehicle_id            trip_id route_id        lat        lon  \
0  9031001001004806  14010000724326947           59.307518  18.074780   
1  9031001001004825  14010000686179377           59.307701  18.075348   
2  9031001004505580  14010000721149392           59.752499  18.692451   
3  9031001003005394  14010000723497915           59.315498  18.085537   
4  9031001003005037  14010000597114396           59.144562  18.126465   

    timestamp  
0  1783869654  
1  1783869654  
2  1783869653  
3  1783869654  
4  1783869654  
891


In [7]:
vehicles = (
    realtime
    .merge(
        routes[
            ["route_id",
             "agency_id",
             "route_short_name",
             "route_type"]
        ],
        on="route_id",
        how="left"
    )
    .merge(
        agencies[
            ["agency_id",
             "agency_name"]
        ],
        on="agency_id",
        how="left"
    )
)

vehicles.head()

,vehicle_id,trip_id,route_id,lat,lon,timestamp,agency_id,route_short_name,route_type,agency_name
0,9031001001004806,14010000724326947,,59.307518,18.074780,1783869654,NaN,NaN,NaN,NaN
1,9031001001004825,14010000686179377,,59.307701,18.075348,1783869654,NaN,NaN,NaN,NaN
2,9031001004505580,14010000721149392,,59.752499,18.692451,1783869653,NaN,NaN,NaN,NaN
3,9031001003005394,14010000723497915,,59.315498,18.085537,1783869654,NaN,NaN,NaN,NaN
4,9031001003005037,14010000597114396,,59.144562,18.126465,1783869654,NaN,NaN,NaN,NaN


In [8]:
(
    vehicles
    .groupby(["agency_name"])
    .size()
    .sort_values(ascending=False)
)

agency_name
AB SL Kundtjänst    9
dtype: int64

In [9]:
(
    vehicles
    .groupby(["agency_id","agency_name"])
    .size()
    .reset_index(name="vehicles")
    .sort_values("vehicles",ascending=False)
)

,agency_id,agency_name,vehicles
0,505000000000000001,AB SL Kundtjänst,9
